# Neural owner head: training on Colab

Trains `src/owner_nn.py` on the exported **DFL** bench (CC BY 4.0): 7 leave-one-match-out models plus 1 final model on all 7 matches, **each with 3 seeds** whose log-odds are averaged.

**Rules:** only `colab_bundle.zip` (DFL bench + code) comes in here. Nothing from the challenge: no ATD, video, Smart Tagging or homography. **Keep Gemini disabled** in this notebook.

1. *Runtime → Change runtime type → GPU* (A100 if available; a T4 works, more slowly).
2. Build the bundle locally with `python scripts/export_nn_bench.py` and upload `external/nn/colab_bundle.zip` to `MyDrive/e4/` (or upload it in cell 2).
3. Run the cells in order. Cell 4 can be re-run if Colab disconnects: it resumes by fold and seed.
4. Download `oof_rnn.npz`, `owner_nn_s0.pt`, `owner_nn_s1.pt`, `owner_nn_s2.pt` and `owner_nn.json`, and put them in `external/nn/out/` locally (the `.pt` files and the `.json` go to `models/`).

**1. GPU and Drive.** Results are written to your Drive, so they survive a disconnect.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/e4'
import os; os.makedirs(WORK, exist_ok=True); print(os.listdir(WORK))

**2. Bundle.** Asks for an upload only if the zip is not already in `MyDrive/e4`.

In [ ]:
import os, shutil
if not os.path.exists(f'{WORK}/colab_bundle.zip'):
    from google.colab import files
    up = files.upload()
    shutil.move(next(iter(up)), f'{WORK}/colab_bundle.zip')
!rm -rf /content/e4 && mkdir -p /content/e4 && unzip -q {WORK}/colab_bundle.zip -d /content/e4 && ls -la /content/e4 /content/e4/src

**3. Smoke test** (~1–2 min): the model's exact team symmetry, and one tiny fold end to end.

In [ ]:
%cd /content/e4
!python -m pytest -q tests/test_nn_inputs.py
!python -m src.owner_nn --data nn_bench.npz --out /content/smoke --smoke

**4. Training:** 7 folds + the final model, 3 seeds each (24 runs, ~70 min on an A100). Resumable: it skips whatever is already in `WORK/out`.

In [ ]:
%cd /content/e4
!python -m src.owner_nn --data nn_bench.npz --out {WORK}/out 2>&1 | tee -a {WORK}/out_log.txt

**5. Download** the outputs and the log.

In [ ]:
from google.colab import files
for f in ('oof_rnn.npz', 'owner_nn_s0.pt', 'owner_nn_s1.pt', 'owner_nn_s2.pt', 'owner_nn.json'):
    files.download(f'{WORK}/out/{f}')
files.download(f'{WORK}/out_log.txt')